# Step 7: Data Cleaning

## 7.1 Loading Audit Data

Import the audit-stage datasets exported from the data loading and audit notebook.
The original CSV files will remain unchanged.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
# Project path
PROJECT_DIR = Path(
    r"C:\Users\Yaseen\DA_DS\MINI_PROJECT\smart_data_analysis"
)

PROCESSED_DIR = PROJECT_DIR / "processed_data"

In [6]:
# Import audit-stage datasets
sales = pd.read_parquet(
    PROCESSED_DIR / "sales_audit.parquet"
)

In [7]:
inventory = pd.read_parquet(
    PROCESSED_DIR / "inventory_audit.parquet"
)

In [8]:
print("Audit data imported successfully!")
print("Sales Shape:", sales.shape)
print("Inventory Shape:", inventory.shape)

Audit data imported successfully!
Sales Shape: (125751, 17)
Inventory Shape: (284755, 18)


In [9]:
print("SALES DATA TYPES")
display(sales.dtypes)

print("\nINVENTORY DATA TYPES")
display(inventory.dtypes)

SALES DATA TYPES


Transaction Date          datetime64[us]
Sales Type                           str
Is Return                          int64
Reason of Return                 float64
Supplier                             str
Product No                           str
Product Description                  str
Product Division                     str
Product Category                     str
Product Subcategory                  str
Product Segment                      str
Store                                str
Sales Channel                        str
Qty Sold                         float64
Sales Amount                     float64
Cogs                             float64
Number of Transactions             int64
dtype: object


INVENTORY DATA TYPES


Start Date                  datetime64[us]
End Date                    datetime64[us]
Stock Status                           str
Supplier                               str
Product No                             str
Product Description                    str
Product Division                       str
Product Category                       str
Product Subcategory                    str
Product Segment                        str
Store                                  str
Store Type                             str
Sales Channel                          str
Qty on hand                          int64
Stocks Selling Amount              float64
Cost of Stocks                     float64
Stock Unit Selling Price           float64
Stock Unit Cost Price              float64
dtype: object

In [11]:
print("SALES MISSING VALUES")
display(sales.isnull().sum()[sales.isnull().sum() > 0])

SALES MISSING VALUES


Reason of Return    125751
dtype: int64

In [12]:
print("\nINVENTORY MISSING VALUES")
display(
    inventory.isnull().sum()[
        inventory.isnull().sum() > 0
    ]
)


INVENTORY MISSING VALUES


Series([], dtype: int64)

In [14]:
print("\nSales duplicate rows:", sales.duplicated().sum())
print("Inventory duplicate rows:", inventory.duplicated().sum())


Sales duplicate rows: 0
Inventory duplicate rows: 0


## Step 7.2 — 100% missing column handle karna

In [16]:
missing_count = sales["Reason of Return"].isna().sum()
total_rows = len(sales)

missing_percentage = (missing_count / total_rows) * 100

In [17]:
print("Missing values:", missing_count)
print("Total rows:", total_rows)
print("Missing percentage:", missing_percentage, "%")

Missing values: 125751
Total rows: 125751
Missing percentage: 100.0 %


In [18]:
sales_clean = sales.copy()
inventory_clean = inventory.copy()

In [19]:
# Drop the completely empty column from the cleaning copy
sales_clean = sales_clean.drop(
    columns=["Reason of Return"]
)

In [20]:
print("Original sales shape:", sales.shape)
print("Clean sales shape:", sales_clean.shape)
print("Inventory shape:", inventory_clean.shape)

Original sales shape: (125751, 17)
Clean sales shape: (125751, 16)
Inventory shape: (284755, 18)


## 7.3 Investigating Negative Inventory Values

Negative stock quantities and financial values are investigated before making any cleaning decisions.

In [21]:
# Count negative values
print("Negative stock quantities:",
      (inventory_clean["Qty on hand"] < 0).sum())

print("Negative stock selling amounts:",
      (inventory_clean["Stocks Selling Amount"] < 0).sum())

print("Negative cost of stocks:",
      (inventory_clean["Cost of Stocks"] < 0).sum())

Negative stock quantities: 1380
Negative stock selling amounts: 1380
Negative cost of stocks: 8022


In [22]:
negative_cost = inventory_clean[
    inventory_clean["Cost of Stocks"] < 0
]

display(
    negative_cost[
        [
            "Product No",
            "Product Description",
            "Store",
            "Stock Status",
            "Qty on hand",
            "Stocks Selling Amount",
            "Cost of Stocks",
            "Stock Unit Cost Price"
        ]
    ].head(10)
)

,Product No,Product Description,Store,Stock Status,Qty on hand,Stocks Selling Amount,Cost of Stocks,Stock Unit Cost Price
5,PROD-124597,PROD-100989 SEG-500125 Toddler Footwear Flight...,STR-1163,Full Price,0,0.0,-6.80,0.00
11,PROD-100353,PROD-164489 SEG-515310 Femme Footwear Court Ca...,STR-1145,Full Price,-1,-120.0,-59.17,59.17
13,PROD-138976,PROD-122134 SEG-504485 Junior Apparel Cozy Tro...,STR-1010,Full Price,0,0.0,-0.15,0.00
14,PROD-165960,PROD-149190 SEG-511431 Scholar Footwear Trailb...,STR-1075,Full Price,0,0.0,-1.63,0.00
43,PROD-166159,PROD-149169 SEG-511428 Femme Footwear Trailbla...,STR-1085,Full Price,0,0.0,-0.01,0.00
44,PROD-169472,PROD-153437 SEG-512518 Femme Footwear Court Ca...,STR-1237,Full Price,0,0.0,-0.01,0.00
69,PROD-124436,PROD-154331 SEG-512711 Femme Footwear Trailbla...,STR-1085,Full Price,0,0.0,-0.06,0.00
143,PROD-152045,PROD-166020 SEG-515561 Scholar Footwear Modern...,STR-1010,Markdown Tier 2,0,0.0,-0.03,0.00
207,PROD-139829,PROD-138298 SEG-508706 Scholar Footwear Modern...,STR-1145,Full Price,0,0.0,-1.40,0.00
239,PROD-124433,PROD-154328 SEG-512711 Femme Footwear Trailbla...,STR-1085,Full Price,0,0.0,-0.05,0.00


In [23]:
# Categorize negative cost records
negative_cost = inventory_clean[
    inventory_clean["Cost of Stocks"] < 0
]

print("Total negative cost records:", len(negative_cost))

print("\nQuantity status among negative cost records:")

display(
    negative_cost.assign(
        Quantity_Group=np.select(
            [
                negative_cost["Qty on hand"] < 0,
                negative_cost["Qty on hand"] == 0,
                negative_cost["Qty on hand"] > 0
            ],
            [
                "Negative Quantity",
                "Zero Quantity",
                "Positive Quantity"
            ],
            default="Unknown"
        )
    )
    .groupby("Quantity_Group")
    .agg(
        records=("Cost of Stocks", "size"),
        total_cost=("Cost of Stocks", "sum")
    )
    .sort_values("records", ascending=False)
)

Total negative cost records: 8022

Quantity status among negative cost records:


,records,total_cost
Quantity_Group,,
Zero Quantity,6625,-11716.43
Negative Quantity,1376,-65371.33
Positive Quantity,21,-130.44


In [24]:
# Negative stock cost with zero quantity
zero_qty_negative_cost = inventory_clean[
    (inventory_clean["Qty on hand"] == 0) &
    (inventory_clean["Cost of Stocks"] < 0)
]

print("Records:", len(zero_qty_negative_cost))

print(
    "Negative cost with zero unit cost:",
    (
        zero_qty_negative_cost["Stock Unit Cost Price"] == 0
    ).sum()
)

print(
    "Negative cost with positive unit cost:",
    (
        zero_qty_negative_cost["Stock Unit Cost Price"] > 0
    ).sum()
)

display(
    zero_qty_negative_cost[
        [
            "Product No",
            "Store",
            "Stock Status",
            "Qty on hand",
            "Stocks Selling Amount",
            "Cost of Stocks",
            "Stock Unit Cost Price"
        ]
    ].head(15)
)

Records: 6625
Negative cost with zero unit cost: 6625
Negative cost with positive unit cost: 0


,Product No,Store,Stock Status,Qty on hand,Stocks Selling Amount,Cost of Stocks,Stock Unit Cost Price
5,PROD-124597,STR-1163,Full Price,0,0.0,-6.80,0.0
13,PROD-138976,STR-1010,Full Price,0,0.0,-0.15,0.0
14,PROD-165960,STR-1075,Full Price,0,0.0,-1.63,0.0
43,PROD-166159,STR-1085,Full Price,0,0.0,-0.01,0.0
44,PROD-169472,STR-1237,Full Price,0,0.0,-0.01,0.0
69,PROD-124436,STR-1085,Full Price,0,0.0,-0.06,0.0
143,PROD-152045,STR-1010,Markdown Tier 2,0,0.0,-0.03,0.0
207,PROD-139829,STR-1145,Full Price,0,0.0,-1.40,0.0
239,PROD-124433,STR-1085,Full Price,0,0.0,-0.05,0.0
302,PROD-161834,STR-1006,Full Price,0,0.0,-0.01,0.0


## Step 7.4 — Negative cost records ki further investigation

In [25]:
# Investigate negative Cost of Stocks by Stock Status

negative_cost = inventory_clean[
    inventory_clean["Cost of Stocks"] < 0
]

negative_cost_summary = (
    negative_cost
    .groupby("Stock Status")
    .agg(
        records=("Cost of Stocks", "size"),
        total_negative_cost=("Cost of Stocks", "sum"),
        zero_quantity_records=(
            "Qty on hand",
            lambda x: (x == 0).sum()
        ),
        negative_quantity_records=(
            "Qty on hand",
            lambda x: (x < 0).sum()
        ),
        positive_quantity_records=(
            "Qty on hand",
            lambda x: (x > 0).sum()
        )
    )
    .sort_values("records", ascending=False)
)

display(negative_cost_summary)

,records,total_negative_cost,zero_quantity_records,negative_quantity_records,positive_quantity_records
Stock Status,,,,,
Full Price,7075,-65903.68,5980,1075,20
Markdown Tier 1,467,-5574.12,306,160,1
Markdown Tier 2,399,-4154.43,318,81,0
Clearance,81,-1585.97,21,60,0


In [26]:
# Compare negative quantity and negative cost

negative_quantity = inventory_clean["Qty on hand"] < 0
negative_cost = inventory_clean["Cost of Stocks"] < 0

print("Negative quantity records:", negative_quantity.sum())
print("Negative cost records:", negative_cost.sum())

print(
    "Both negative:",
    (negative_quantity & negative_cost).sum()
)

print(
    "Negative quantity but non-negative cost:",
    (negative_quantity & ~negative_cost).sum()
)

print(
    "Negative cost but non-negative quantity:",
    (negative_cost & ~negative_quantity).sum()
)

Negative quantity records: 1380
Negative cost records: 8022
Both negative: 1376
Negative quantity but non-negative cost: 4
Negative cost but non-negative quantity: 6646


## Step 8.1 — Cleaning rules apply Apply

In [27]:
# 1. Sales: remove fully missing return-reason column
# Already done in Step 7.2; this is a safety check.
if "Reason of Return" in sales_clean.columns:
    sales_clean = sales_clean.drop(columns=["Reason of Return"])


In [28]:
# 2. Inventory: preserve original numeric values
# Add flags instead of deleting suspicious records.
inventory_clean["Negative_Quantity_Flag"] = (
    inventory_clean["Qty on hand"] < 0
)

inventory_clean["Negative_Cost_Flag"] = (
    inventory_clean["Cost of Stocks"] < 0
)

inventory_clean["Negative_Quantity_And_Cost_Flag"] = (
    inventory_clean["Negative_Quantity_Flag"]
    & inventory_clean["Negative_Cost_Flag"]
)

In [29]:
# 3. Verify final shapes and flags
print("Sales cleaned shape:", sales_clean.shape)
print("Inventory cleaned shape:", inventory_clean.shape)

print(
    "\nNegative quantity flags:",
    inventory_clean["Negative_Quantity_Flag"].sum()
)

print(
    "Negative cost flags:",
    inventory_clean["Negative_Cost_Flag"].sum()
)

print(
    "Both negative flags:",
    inventory_clean["Negative_Quantity_And_Cost_Flag"].sum()
)

print("\nRemaining missing values in sales:")
display(sales_clean.isnull().sum()[sales_clean.isnull().sum() > 0])

print("\nRemaining missing values in inventory:")
display(
    inventory_clean.isnull().sum()[
        inventory_clean.isnull().sum() > 0
    ]
)

Sales cleaned shape: (125751, 16)
Inventory cleaned shape: (284755, 21)

Negative quantity flags: 1380
Negative cost flags: 8022
Both negative flags: 1376

Remaining missing values in sales:


Series([], dtype: int64)


Remaining missing values in inventory:


Series([], dtype: int64)

## Step 8.2 — Final cleaned data save karna

In [30]:
from pathlib import Path

PROCESSED_DIR = PROJECT_DIR / "processed_data"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sales_clean.to_parquet(
    PROCESSED_DIR / "sales_clean.parquet",
    index=False
)

inventory_clean.to_parquet(
    PROCESSED_DIR / "inventory_clean.parquet",
    index=False
)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


In [31]:
# ==========================================
# STEP 8.2: FINAL DATA VALIDATION
# ==========================================

print("========== SALES VALIDATION ==========")

print("Shape:", sales_clean.shape)
print("Duplicate rows:", sales_clean.duplicated().sum())
print("Missing values:", sales_clean.isnull().sum().sum())

print("\nSales date range:")
print("Start:", sales_clean["Transaction Date"].min())
print("End:", sales_clean["Transaction Date"].max())

print("\n========== INVENTORY VALIDATION ==========")

print("Shape:", inventory_clean.shape)
print("Duplicate rows:", inventory_clean.duplicated().sum())
print("Missing values:", inventory_clean.isnull().sum().sum())

print("\nInventory date ranges:")
print("Start Date:", inventory_clean["Start Date"].min())
print("End Date:", inventory_clean["End Date"].max())

print("\n========== FLAG VALIDATION ==========")

print(
    "Negative quantity flag matches:",
    (
        inventory_clean["Negative_Quantity_Flag"]
        == (inventory_clean["Qty on hand"] < 0)
    ).all()
)

print(
    "Negative cost flag matches:",
    (
        inventory_clean["Negative_Cost_Flag"]
        == (inventory_clean["Cost of Stocks"] < 0)
    ).all()
)

print(
    "Combined flag matches:",
    (
        inventory_clean["Negative_Quantity_And_Cost_Flag"]
        == (
            (inventory_clean["Qty on hand"] < 0)
            & (inventory_clean["Cost of Stocks"] < 0)
        )
    ).all()
)

========== SALES VALIDATION ==========
Shape: (125751, 16)
Duplicate rows: 0
Missing values: 0

Sales date range:
Start: 2025-06-01 00:00:00
End: 2026-04-24 00:00:00

========== INVENTORY VALIDATION ==========
Shape: (284755, 21)
Duplicate rows: 0
Missing values: 0

Inventory date ranges:
Start Date: 2025-06-01 00:00:00
End Date: 9999-12-31 00:00:00

========== FLAG VALIDATION ==========
Negative quantity flag matches: True
Negative cost flag matches: True
Combined flag matches: True


In [32]:
# ==========================================
# STEP 8.3: SAVE CLEANED DATASETS
# ==========================================

PROCESSED_DIR = PROJECT_DIR / "processed_data"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sales_clean.to_parquet(
    PROCESSED_DIR / "sales_clean.parquet",
    index=False
)

inventory_clean.to_parquet(
    PROCESSED_DIR / "inventory_clean.parquet",
    index=False
)

print("Sales cleaned file saved.")
print("Inventory cleaned file saved.")
print("Location:", PROCESSED_DIR)

Sales cleaned file saved.
Inventory cleaned file saved.
Location: C:\Users\Yaseen\DA_DS\MINI_PROJECT\smart_data_analysis\processed_data
